In [ ]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("PairRDDs")
  .master("local[*]")
  .getOrCreate()

val sc = spark.sparkContext

// RDD de ventas como cadenas: "producto,región,importe"
val ventas = sc.parallelize(List(
  "Laptop,Norte,1200",
  "Teclado,Sur,45",
  "Monitor,Norte,350",
  "Laptop,Sur,1200",
  "Ratón,Norte,25",
  "Monitor,Sur,350",
  "Teclado,Norte,45"
))

// Convertimos a Pair RDD: (región, importe)
val ventasPorRegion = ventas.map { linea =>
  val campos = linea.split(",")
  val region  = campos(1)
  val importe = campos(2).toDouble
  (region, importe)   // ← tupla (clave, valor)
}

ventasPorRegion.collect().foreach(println)
// (Norte,1200.0)
// (Sur,45.0)
// (Norte,350.0)
// (Sur,1200.0)
// (Norte,25.0)
// (Sur,350.0)
// (Norte,45.0)

// Objetivo: para cada región → List con todos los importes
val importesPorRegion = ventasPorRegion.combineByKey(
  (v: Double) => List(v),                       // createCombiner: primer valor → List
  (acc: List[Double], v: Double) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

importesPorRegion.collect().foreach { case (region, lista) =>
  println(s"$region → $lista")
}
// Norte → List(1200.0, 350.0, 25.0, 45.0)
// Sur   → List(45.0, 1200.0, 350.0)

In [2]:
def verParticiones[T](rdd: org.apache.spark.rdd.RDD[T]): Unit = {
  rdd.mapPartitionsWithIndex { (index, iter) =>
    Iterator(s"📦 Partición $index -> ${iter.toList.mkString(", ")}")
  }.collect().foreach(println)
}

defined function verParticiones

🧪 Ejercicio 1: Temperaturas por ciudad

In [3]:
val temperaturas = sc.parallelize(List(
  ("Madrid", 18.5),
  ("Barcelona", 20.0),
  ("Madrid", 21.0),
  ("Valencia", 22.5),
  ("Sevilla", 25.0),
  ("Barcelona", 19.5),
  ("Madrid", 17.0),
  ("Valencia", 23.0),
  ("Sevilla", 26.5),
  ("Barcelona", 18.0),
  ("Madrid", 20.5),
  ("Valencia", 21.5)
), 3)

verParticiones(temperaturas)

2026-04-27T07:53:12.767444600Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

temperaturas: org.apache.spark.rdd.RDD[(String, Double)] = ParallelCollectionRDD[3] at parallelize at cmd3.sc:1

In [4]:
// objetivo para cada ciudad → List con todas las temperaturas
val tempPorCiudad = temperaturas.combineByKey(
  (v: Double) => List(v),                       // createCombiner: primer valor → List
  (acc: List[Double], v: Double) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

tempPorCiudad.collect().foreach { case (ciudad, lista) =>
  println(s"$ciudad → $lista")
}

Barcelona → List(20.0, 19.5, 18.0)
Sevilla → List(25.0, 26.5)
Valencia → List(22.5, 23.0, 21.5)
Madrid → List(18.5, 21.0, 17.0, 20.5)


tempPorCiudad: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[5] at combineByKey at cmd4.sc:5

Ejercicio 2: Productos comprados por cliente

In [5]:
val compras = sc.parallelize(List(
  ("Ana", "Laptop"),
  ("Luis", "Teclado"),
  ("Ana", "Mouse"),
  ("Marta", "Monitor"),
  ("Luis", "Tablet"),
  ("Ana", "Auriculares"),
  ("Carlos", "Impresora"),
  ("Marta", "Webcam"),
  ("Luis", "Ratón"),
  ("Carlos", "SSD"),
  ("Ana", "USB"),
  ("Marta", "Silla")
), 3)

verParticiones(compras)

2026-04-27T08:26:24.492432100Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

compras: org.apache.spark.rdd.RDD[(String, String)] = ParallelCollectionRDD[6] at parallelize at cmd5.sc:1

In [ ]:
// objetivo para cada cliente → List con todas los productos    
val productosCliente = compras.combineByKey(
  (v: String) => List(v),                       // createCombiner: primer valor → List
  (acc: List[String], v: String) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[String], acc2: List[String]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

productosCliente.collect().foreach { case (cliente, lista) =>
  println(s"$cliente → $lista")
}

Marta → List(Monitor, Webcam, Silla)
Carlos → List(Impresora, SSD)
Ana → List(Laptop, Mouse, Auriculares, USB)
Luis → List(Teclado, Tablet, Ratón)


productosCliente: org.apache.spark.rdd.RDD[(String, List[String])] = ShuffledRDD[8] at combineByKey at cmd6.sc:5

🧪 Ejercicio 3: Notas por estudiante

In [7]:
val notas = sc.parallelize(List(
  ("Carlos", 8.5),
  ("Marta", 7.0),
  ("Carlos", 9.0),
  ("Ana", 6.5),
  ("Marta", 8.0),
  ("Ana", 7.5),
  ("Luis", 5.5),
  ("Carlos", 10.0),
  ("Luis", 6.0),
  ("Marta", 9.0),
  ("Ana", 8.5),
  ("Luis", 7.0)
), 3)

verParticiones(notas)

📦 Partición 0 -> (Carlos,8.5), (Marta,7.0), (Carlos,9.0), (Ana,6.5)
📦 Partición 1 -> (Marta,8.0), (Ana,7.5), (Luis,5.5), (Carlos,10.0)
📦 Partición 2 -> (Luis,6.0), (Marta,9.0), (Ana,8.5), (Luis,7.0)


notas: org.apache.spark.rdd.RDD[(String, Double)] = ParallelCollectionRDD[9] at parallelize at cmd7.sc:1

In [8]:
// objetivo para cada estudiante → List con todas las notas    
val notasEstudiante = notas.combineByKey(
  (v: Double) => List(v),                       // createCombiner: primer valor → List
  (acc: List[Double], v: Double) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

notasEstudiante.collect().foreach { case (estudiante, lista) =>
  println(s"$estudiante → $lista")
}

Marta → List(7.0, 8.0, 9.0)
Carlos → List(8.5, 9.0, 10.0)
Ana → List(6.5, 7.5, 8.5)
Luis → List(5.5, 6.0, 7.0)


notasEstudiante: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[11] at combineByKey at cmd8.sc:5

🧪 Ejercicio 4: Palabras por inicial

In [9]:
val palabras = sc.parallelize(List(
  ("A", "Azure"),
  ("S", "Spark"),
  ("A", "Almond"),
  ("S", "Scala"),
  ("P", "Python"),
  ("J", "Java"),
  ("A", "Airflow"),
  ("S", "SQL"),
  ("P", "PostgreSQL"),
  ("J", "Jupyter"),
  ("A", "AWS"),
  ("P", "PySpark")
), 3)

verParticiones(palabras)

📦 Partición 0 -> (A,Azure), (S,Spark), (A,Almond), (S,Scala)
📦 Partición 1 -> (P,Python), (J,Java), (A,Airflow), (S,SQL)
📦 Partición 2 -> (P,PostgreSQL), (J,Jupyter), (A,AWS), (P,PySpark)


palabras: org.apache.spark.rdd.RDD[(String, String)] = ParallelCollectionRDD[12] at parallelize at cmd9.sc:1

In [10]:
// objetivo para cada letra → List con todas las palabras   
val palabrasLetra = palabras.combineByKey(
  (v: String) => List(v),                       // createCombiner: primer valor → List
  (acc: List[String], v: String) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[String], acc2: List[String]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

palabrasLetra.collect().foreach { case (letra, lista) =>
  println(s"$letra → $lista")
}

P → List(Python, PostgreSQL, PySpark)
A → List(Azure, Almond, Airflow, AWS)
J → List(Java, Jupyter)
S → List(Spark, Scala, SQL)


palabrasLetra: org.apache.spark.rdd.RDD[(String, List[String])] = ShuffledRDD[14] at combineByKey at cmd10.sc:5

🧪 Caso de Estudio: Análisis de Actividad de Usuarios en una Plataforma Digital

usuario, tipo_evento, duracion_minutos


In [11]:
val eventos = sc.parallelize(List(
  ("user1", "video", 10.0),
  ("user2", "quiz", 5.0),
  ("user1", "video", 15.0),
  ("user3", "video", 20.0),
  ("user2", "video", 8.0),
  ("user1", "quiz", 7.0),
  ("user3", "quiz", 6.0),
  ("user2", "video", 12.0),
  ("user1", "video", 9.0),
  ("user3", "video", 11.0),
  ("user2", "quiz", 4.0),
  ("user3", "video", 13.0)
), 3)

eventos: org.apache.spark.rdd.RDD[(String, String, Double)] = ParallelCollectionRDD[15] at parallelize at cmd11.sc:1

In [12]:
verParticiones(eventos)

2026-04-27T08:32:10.320134800Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

In [14]:
// Convertimos a Pair RDD: (usuario, duracion)
val eventoUsuario = eventos.map { case (usuario, evento, duracion_minutos) =>
  (usuario, duracion_minutos)   // ← tupla (clave, valor)
}


// Objetivo: para cada usuario → List con todas las duraciones
val usuarioDuracion = eventoUsuario.combineByKey(
  (v: Double) => List(v),                       // createCombiner: primer valor → List
  (acc: List[Double], v: Double) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

usuarioDuracion.collect().foreach { case (usuario, lista) =>
  println(s"$usuario → $lista")
}

user3 → List(20.0, 6.0, 11.0, 13.0)
user1 → List(10.0, 15.0, 7.0, 9.0)
user2 → List(5.0, 8.0, 12.0, 4.0)


eventoUsuario: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[19] at map at cmd14.sc:2
usuarioDuracion: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[20] at combineByKey at cmd14.sc:11

In [17]:
val totalUsuario = usuarioDuracion.mapValues(lista => lista.sum)

totalUsuario.collect().foreach { case (usuario, total) =>
  println(s"$usuario → $total")
}


user3 → 50.0
user1 → 41.0
user2 → 29.0


totalUsuario: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[24] at mapValues at cmd17.sc:1

In [16]:
val promedioUsuario = usuarioDuracion.mapValues(lista => lista.sum / lista.size)

promedioUsuario.collect().foreach { case (usuario, promedio) =>
  println(s"$usuario → $promedio")
}

user3 → 12.5
user1 → 10.25
user2 → 7.25


promedioUsuario: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[23] at mapValues at cmd16.sc:1

Sesión 2 - Spark DataFrames I: Introducción

In [18]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import $ivy.`org.apache.spark::spark-core:4.1.1`
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("DataFrames_Dia14")   // nombre visible en la Spark UI
  .master("local[*]")            // modo local, todos los cores disponibles
  .getOrCreate()                 // crea o reutiliza una sesión existente

// Desde SparkSession podemos acceder al SparkContext si lo necesitamos
val sc = spark.sparkContext

println(s"Spark ${spark.version} listo")

2026-04-27T10:36:59.582473200Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@a9c4c1c
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@4a5c9fd2

In [19]:
import spark.implicits._
// Forma 1: Seq de tuplas + toDF con nombres de columna
val dfEmpleados = Seq(
  (1, "Ana García",    28, "Ingeniería"),
  (2, "Luis Martínez", 34, "Marketing"),
  (3, "Marta López",   22, "Ingeniería"),
  (4, "Pedro Ruiz",    41, "Dirección")
).toDF("id", "nombre", "edad", "departamento")

dfEmpleados.show()
// +---+-------------+----+------------+
// | id|       nombre|edad|departamento|
// +---+-------------+----+------------+
// |  1|   Ana García|  28|  Ingeniería|
// |  2|Luis Martínez|  34|   Marketing|
// |  3|  Marta López|  22|  Ingeniería|
// |  4|   Pedro Ruiz|  41|   Dirección|
// +---+-------------+----+------------+

+---+-------------+----+------------+
| id|       nombre|edad|departamento|
+---+-------------+----+------------+
|  1|   Ana García|  28|  Ingeniería|
|  2|Luis Martínez|  34|   Marketing|
|  3|  Marta López|  22|  Ingeniería|
|  4|   Pedro Ruiz|  41|   Dirección|
+---+-------------+----+------------+



import spark.implicits._
dfEmpleados: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 2 more fields]

In [20]:
import spark.implicits._

import spark.implicits._

In [21]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val ruta = "C:/Notebooks/datos"
Files.createDirectories(Paths.get(ruta))

val contenidoCSV =
"""id_venta,fecha,producto,categoria,cantidad,precio_unitario,ciudad
1,2026-04-01,Portátil,Informática,2,850.50,Madrid
2,2026-04-01,Ratón,Informática,5,18.90,Valencia
3,2026-04-02,Teclado,Informática,3,45.00,Sevilla
4,2026-04-02,Monitor,Informática,1,199.99,Madrid
5,2026-04-03,Silla,Oficina,4,120.00,Barcelona
6,2026-04-03,Mesa,Oficina,2,250.00,Zaragoza
7,2026-04-04,Webcam,Informática,6,39.90,Madrid
8,2026-04-04,Auriculares,Informática,3,59.99,Valencia
"""

val pathCSV = s"$ruta/ventas.csv"

Files.write(
  Paths.get(pathCSV),
  contenidoCSV.getBytes(StandardCharsets.UTF_8)
)

println(s"CSV creado en: $pathCSV")

CSV creado en: C:/Notebooks/datos/ventas.csv


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
ruta: String = "C:/Notebooks/datos"
res21_3: java.nio.file.Path = C:\Notebooks\datos
contenidoCSV: String = """id_venta,fecha,producto,categoria,cantidad,precio_unitario,ciudad
1,2026-04-01,Portátil,Informática,2,850.50,Madrid
2,2026-04-01,Ratón,Informática,5,18.90,Valencia
3,2026-04-02,Teclado,Informática,3,45.00,Sevilla
4,2026-04-02,Monitor,Informática,1,199.99,Madrid
5,2026-04-03,Silla,Oficina,4,120.00,Barcelona
6,2026-04-03,Mesa,Oficina,2,250.00,Zaragoza
7,2026-04-04,Webcam,Informática,6,39.90,Madrid
8,2026-04-04,Auriculares,Informática,3,59.99,Valencia
"""
pathCSV: String = "C:/Notebooks/datos/ventas.csv"
res21_6: java.nio.file.Path = C:\Notebooks\datos\ventas.csv

In [22]:
val dfVentas = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("C:/Notebooks/datos/ventas.csv")

dfVentas.show(5)
dfVentas.printSchema()

+--------+----------+--------+-----------+--------+---------------+---------+
|id_venta|     fecha|producto|  categoria|cantidad|precio_unitario|   ciudad|
+--------+----------+--------+-----------+--------+---------------+---------+
|       1|2026-04-01|Portátil|Informática|       2|          850.5|   Madrid|
|       2|2026-04-01|   Ratón|Informática|       5|           18.9| Valencia|
|       3|2026-04-02| Teclado|Informática|       3|           45.0|  Sevilla|
|       4|2026-04-02| Monitor|Informática|       1|         199.99|   Madrid|
|       5|2026-04-03|   Silla|    Oficina|       4|          120.0|Barcelona|
+--------+----------+--------+-----------+--------+---------------+---------+
only showing top 5 rows
root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- ciudad: string (nul

dfVentas: org.apache.spark.sql.package.DataFrame = [id_venta: int, fecha: date ... 5 more fields]

In [24]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val ruta = "C:/Notebooks/datos" // Usa tu propia ruta, sino te dará error
Files.createDirectories(Paths.get(ruta))

val jsonSimple =
"""{"id":1,"nombre":"Ana García","edad":28,"departamento":"Ingeniería"}
{"id":2,"nombre":"Luis Martínez","edad":34,"departamento":"Marketing"}
{"id":3,"nombre":"Marta López","edad":22,"departamento":"Ingeniería"}
{"id":4,"nombre":"Pedro Ruiz","edad":41,"departamento":"Dirección"}"""

val rutaJsonSimple = s"$ruta/empleados_simple.json"

Files.write(
  Paths.get(rutaJsonSimple),
  jsonSimple.getBytes(StandardCharsets.UTF_8)
)

println(s"JSON simple creado en: $rutaJsonSimple")

JSON simple creado en: C:/Notebooks/datos/empleados_simple.json


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
ruta: String = "C:/Notebooks/datos"
res24_3: java.nio.file.Path = C:\Notebooks\datos
jsonSimple: String = """{"id":1,"nombre":"Ana García","edad":28,"departamento":"Ingeniería"}
{"id":2,"nombre":"Luis Martínez","edad":34,"departamento":"Marketing"}
{"id":3,"nombre":"Marta López","edad":22,"departamento":"Ingeniería"}
{"id":4,"nombre":"Pedro Ruiz","edad":41,"departamento":"Dirección"}"""
rutaJsonSimple: String = "C:/Notebooks/datos/empleados_simple.json"
res24_6: java.nio.file.Path = C:\Notebooks\datos\empleados_simple.json

In [25]:
val dfJsonSimple = spark.read
  .option("inferSchema", "true")
  .json(rutaJsonSimple)

dfJsonSimple.show(false)
dfJsonSimple.printSchema()

2026-04-27T10:44:11.277887400Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

dfJsonSimple: org.apache.spark.sql.package.DataFrame = [departamento: string, edad: bigint ... 2 more fields]

In [26]:
val jsonMultilinea =
"""[
  {
    "id": 1,
    "nombre": "Ana García",
    "edad": 28,
    "departamento": "Ingeniería"
  },
  {
    "id": 2,
    "nombre": "Luis Martínez",
    "edad": 34,
    "departamento": "Marketing"
  },
  {
    "id": 3,
    "nombre": "Marta López",
    "edad": 22,
    "departamento": "Ingeniería"
  },
  {
    "id": 4,
    "nombre": "Pedro Ruiz",
    "edad": 41,
    "departamento": "Dirección"
  }
]"""

val rutaJsonMultilinea = s"$ruta/empleados_multilinea.json"

Files.write(
  Paths.get(rutaJsonMultilinea),
  jsonMultilinea.getBytes(StandardCharsets.UTF_8)
)

println(s"JSON multilínea creado en: $rutaJsonMultilinea")

JSON multilínea creado en: C:/Notebooks/datos/empleados_multilinea.json


jsonMultilinea: String = """[
  {
    "id": 1,
    "nombre": "Ana García",
    "edad": 28,
    "departamento": "Ingeniería"
  },
  {
    "id": 2,
    "nombre": "Luis Martínez",
    "edad": 34,
    "departamento": "Marketing"
  },
  {
    "id": 3,
    "nombre": "Marta López",
    "edad": 22,
    "departamento": "Ingeniería"
  },
  {
    "id": 4,
    "nombre": "Pedro Ruiz",
    "edad": 41,
    "departamento": "Dirección"
  }
]"""
rutaJsonMultilinea: String = "C:/Notebooks/datos/empleados_multilinea.json"
res26_2: java.nio.file.Path = C:\Notebooks\datos\empleados_multilinea.json

In [27]:
val dfJsonMultilinea = spark.read
  .option("multiline", "true")
  .option("inferSchema", "true")
  .json(rutaJsonMultilinea)

dfJsonMultilinea.show(false)
dfJsonMultilinea.printSchema()

+------------+----+---+-------------+
|departamento|edad|id |nombre       |
+------------+----+---+-------------+
|Ingeniería  |28  |1  |Ana García   |
|Marketing   |34  |2  |Luis Martínez|
|Ingeniería  |22  |3  |Marta López  |
|Dirección   |41  |4  |Pedro Ruiz   |
+------------+----+---+-------------+

root
 |-- departamento: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)



dfJsonMultilinea: org.apache.spark.sql.package.DataFrame = [departamento: string, edad: bigint ... 2 more fields]

In [28]:
val dfVentas = spark.read
  .option("header", "true")       // primera fila como nombres de columna
  .option("inferSchema", "true")  // detectar tipos automáticamente
  .csv("C:/Notebooks/datos/ventas.csv")

dfVentas.show(5)         // muestra las primeras 5 filas


+--------+----------+--------+-----------+--------+---------------+---------+
|id_venta|     fecha|producto|  categoria|cantidad|precio_unitario|   ciudad|
+--------+----------+--------+-----------+--------+---------------+---------+
|       1|2026-04-01|Portátil|Informática|       2|          850.5|   Madrid|
|       2|2026-04-01|   Ratón|Informática|       5|           18.9| Valencia|
|       3|2026-04-02| Teclado|Informática|       3|           45.0|  Sevilla|
|       4|2026-04-02| Monitor|Informática|       1|         199.99|   Madrid|
|       5|2026-04-03|   Silla|    Oficina|       4|          120.0|Barcelona|
+--------+----------+--------+-----------+--------+---------------+---------+
only showing top 5 rows


dfVentas: org.apache.spark.sql.package.DataFrame = [id_venta: int, fecha: date ... 5 more fields]

In [29]:
dfVentas.printSchema()   // muestra la estructura con tipos

root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- ciudad: string (nullable = true)



In [30]:
val dfClientes = spark.read
  .option("multiline", "true")    // para JSON con objetos multilínea
  .json("C:/Notebooks/datos/empleados_multilinea.json")

dfClientes.show()

+------------+----+---+-------------+
|departamento|edad| id|       nombre|
+------------+----+---+-------------+
|  Ingeniería|  28|  1|   Ana García|
|   Marketing|  34|  2|Luis Martínez|
|  Ingeniería|  22|  3|  Marta López|
|   Dirección|  41|  4|   Pedro Ruiz|
+------------+----+---+-------------+



dfClientes: org.apache.spark.sql.package.DataFrame = [departamento: string, edad: bigint ... 2 more fields]

In [31]:
dfClientes.printSchema()

root
 |-- departamento: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)



In [34]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

import org.apache.spark.sql.types._

val ruta = "C:/Notebooks/datos"
Files.createDirectories(Paths.get(ruta))

val contenidoPedidos =
"""id_pedido,id_cliente,fecha,importe,completado
1,101,2026-04-01,250.75,true
2,102,2026-04-01,99.90,false
3,103,2026-04-02,430.00,true
4,101,2026-04-03,120.50,true
5,104,2026-04-04,75.25,false
6,105,2026-04-05,680.40,true
"""

val rutaPedidos = s"$ruta/pedidos.csv"

Files.write(
  Paths.get(rutaPedidos),
  contenidoPedidos.getBytes(StandardCharsets.UTF_8)
)

println(s"CSV creado correctamente en: $rutaPedidos")

CSV creado correctamente en: C:/Notebooks/datos/pedidos.csv


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
import org.apache.spark.sql.types._
ruta: String = "C:/Notebooks/datos"
res34_4: java.nio.file.Path = C:\Notebooks\datos
contenidoPedidos: String = """id_pedido,id_cliente,fecha,importe,completado
1,101,2026-04-01,250.75,true
2,102,2026-04-01,99.90,false
3,103,2026-04-02,430.00,true
4,101,2026-04-03,120.50,true
5,104,2026-04-04,75.25,false
6,105,2026-04-05,680.40,true
"""
rutaPedidos: String = "C:/Notebooks/datos/pedidos.csv"
res34_7: java.nio.file.Path = C:\Notebooks\datos\pedidos.csv

In [ ]:
import org.apache.spark.sql.types._
val schemaPedidos = StructType(List(
  StructField("id_pedido", IntegerType, nullable = false),
  StructField("id_cliente", IntegerType, nullable = false),
  StructField("fecha", StringType, nullable = true),
  StructField("importe", DoubleType, nullable = true),
  StructField("completado", BooleanType, nullable = true)
))

val dfPedidos = spark.read
  .option("header", "true")
  .schema(schemaPedidos)
  .csv("C:/Notebooks/datos/pedidos.csv")

dfPedidos.show(false)


In [36]:
dfPedidos.show(false)

+---------+----------+----------+-------+----------+
|id_pedido|id_cliente|fecha     |importe|completado|
+---------+----------+----------+-------+----------+
|1        |101       |2026-04-01|250.75 |true      |
|2        |102       |2026-04-01|99.9   |false     |
|3        |103       |2026-04-02|430.0  |true      |
|4        |101       |2026-04-03|120.5  |true      |
|5        |104       |2026-04-04|75.25  |false     |
|6        |105       |2026-04-05|680.4  |true      |
+---------+----------+----------+-------+----------+



In [37]:
dfPedidos.printSchema()

root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: string (nullable = true)
 |-- importe: double (nullable = true)
 |-- completado: boolean (nullable = true)



In [39]:
import org.apache.spark.sql.types._

val schemaPedidos = StructType(List(
  StructField("id_pedido",   IntegerType, nullable = false),
  StructField("id_cliente",  IntegerType, nullable = false),
  StructField("fecha",       StringType,  nullable = true),
  StructField("importe",     DoubleType,  nullable = true),
  StructField("completado",  BooleanType, nullable = true)
))

val dfPedidos = spark.read
  .option("header", "true")
  .schema(schemaPedidos)          // usamos el schema manual
  .csv("C:/Notebooks/datos/pedidos.csv")

dfPedidos.printSchema()
// root
//  |-- id_pedido: integer (nullable = false)
//  |-- id_cliente: integer (nullable = false)
//  |-- fecha: string (nullable = true)
//  |-- importe: double (nullable = true)
//  |-- completado: boolean (nullable = true)

root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: string (nullable = true)
 |-- importe: double (nullable = true)
 |-- completado: boolean (nullable = true)



import org.apache.spark.sql.types._
schemaPedidos: StructType = Seq(
  StructField(
    name = "id_pedido",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "id_cliente",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "fecha",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "importe",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "completado",
    dataType = BooleanType,
    nullable = true,
    metadata = {}
  )
)
dfPedidos: org.apache.spark.sql.package.DataFrame = [id_pedido: int, id_cliente: int ... 3 more fields]

🗂️ Paso previo: crear los ficheros de datos

[
  {"id": 101, "nombre": "Laptop Pro", "categoria": "Informatica", "precio": 1299.99, "stock": 45},
  {"id": 102, "nombre": "Teclado Inalámbrico", "categoria": "Perifericos", "precio": 59.90, "stock": 120},
  {"id": 103, "nombre": "Monitor 27\"", "categoria": "Monitores", "precio": 349.00, "stock": 30},
  {"id": 104, "nombre": "Ratón Óptico", "categoria": "Perifericos", "precio": 24.95, "stock": 200},
  {"id": 105, "nombre": "Auriculares USB", "categoria": "Audio", "precio": 89.50, "stock": 75},
  {"id": 106, "nombre": "Webcam HD", "categoria": "Perifericos", "precio": 79.00, "stock": 60},
  {"id": 107, "nombre": "Disco SSD 1TB", "categoria": "Almacenamiento", "precio": 119.99, "stock": 90},
  {"id": 108, "nombre": "Hub USB-C", "categoria": "Perifericos", "precio": 44.90, "stock": 150}
]

In [55]:
val jsonMultiProducto = 
"""
[
  {"id": 101, "nombre": "Laptop Pro", "categoria": "Informatica", "precio": 1299.99, "stock": 45},
  {"id": 102, "nombre": "Teclado Inalámbrico", "categoria": "Perifericos", "precio": 59.90, "stock": 120},
  {"id": 103, "nombre": "Monitor 27\"", "categoria": "Monitores", "precio": 349.00, "stock": 30},
  {"id": 104, "nombre": "Ratón Óptico", "categoria": "Perifericos", "precio": 24.95, "stock": 200},
  {"id": 105, "nombre": "Auriculares USB", "categoria": "Audio", "precio": 89.50, "stock": 75},
  {"id": 106, "nombre": "Webcam HD", "categoria": "Perifericos", "precio": 79.00, "stock": 60},
  {"id": 107, "nombre": "Disco SSD 1TB", "categoria": "Almacenamiento", "precio": 119.99, "stock": 90},
  {"id": 108, "nombre": "Hub USB-C", "categoria": "Perifericos", "precio": 44.90, "stock": 150}
]
"""

val ruta = "C:/Notebooks/datos" // Usa tu propia ruta, sino te dará error
Files.createDirectories(Paths.get(ruta))

val rutaProducto = s"$ruta/productos.json"

Files.write(
  Paths.get(rutaProducto),
  jsonMultiProducto.getBytes(StandardCharsets.UTF_8)
)

println(s"JSON multilínea creado en: $rutaProducto")

JSON multilínea creado en: C:/Notebooks/datos/productos.json


jsonMultiProducto: String = """
[
  {"id": 101, "nombre": "Laptop Pro", "categoria": "Informatica", "precio": 1299.99, "stock": 45},
  {"id": 102, "nombre": "Teclado Inalámbrico", "categoria": "Perifericos", "precio": 59.90, "stock": 120},
  {"id": 103, "nombre": "Monitor 27\"", "categoria": "Monitores", "precio": 349.00, "stock": 30},
  {"id": 104, "nombre": "Ratón Óptico", "categoria": "Perifericos", "precio": 24.95, "stock": 200},
  {"id": 105, "nombre": "Auriculares USB", "categoria": "Audio", "precio": 89.50, "stock": 75},
  {"id": 106, "nombre": "Webcam HD", "categoria": "Perifericos", "precio": 79.00, "stock": 60},
  {"id": 107, "nombre": "Disco SSD 1TB", "categoria": "Almacenamiento", "precio": 119.99, "stock": 90},
  {"id": 108, "nombre": "Hub USB-C", "categoria": "Perifericos", "precio": 44.90, "stock": 150}
]
"""
ruta: String = "C:/Notebooks/datos"
res55_2: java.nio.file.Path = C:\Notebooks\datos
rutaProducto: String = "C:/Notebooks/datos/productos.json"
res55_4: java.nio.fi

In [41]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.types._

val spark = SparkSession.builder()
  .appName("Dia14S1_DataFrames")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

import spark.implicits._   // necesario para .toDF() y otras conversiones

spark.sparkContext.setLogLevel("ERROR")

println(s"✅ Spark ${spark.version} listo")

2026-04-27T11:02:39.368330300Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.types._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@a9c4c1c
import spark.implicits._

In [42]:
// Crear DataFrame desde una secuencia de tuplas
val dfCursos = Seq(
  (1, "Big Data con Scala", "Avanzado",  140, 4.8),
  (2, "Python para Datos",  "Intermedio", 80, 4.6),
  (3, "SQL Empresarial",    "Básico",     40, 4.9),
  (4, "Machine Learning",   "Avanzado",  120, 4.7),
  (5, "Power BI",           "Básico",     60, 4.5)
).toDF("id", "titulo", "nivel", "horas", "valoracion")

// Ver las filas
println("=== Contenido del DataFrame ===")
dfCursos.show()

// Ver la estructura
println("=== Schema del DataFrame ===")
dfCursos.printSchema()

// Número de filas y columnas
println(s"Filas: ${dfCursos.count()}")
println(s"Columnas: ${dfCursos.columns.length}")
println(s"Nombres de columnas: ${dfCursos.columns.mkString(", ")}")

=== Contenido del DataFrame ===
+---+------------------+----------+-----+----------+
| id|            titulo|     nivel|horas|valoracion|
+---+------------------+----------+-----+----------+
|  1|Big Data con Scala|  Avanzado|  140|       4.8|
|  2| Python para Datos|Intermedio|   80|       4.6|
|  3|   SQL Empresarial|    Básico|   40|       4.9|
|  4|  Machine Learning|  Avanzado|  120|       4.7|
|  5|          Power BI|    Básico|   60|       4.5|
+---+------------------+----------+-----+----------+

=== Schema del DataFrame ===
root
 |-- id: integer (nullable = false)
 |-- titulo: string (nullable = true)
 |-- nivel: string (nullable = true)
 |-- horas: integer (nullable = false)
 |-- valoracion: double (nullable = false)

Filas: 5
Columnas: 5
Nombres de columnas: id, titulo, nivel, horas, valoracion


dfCursos: org.apache.spark.sql.package.DataFrame = [id: int, titulo: string ... 3 more fields]

In [43]:
// Estadísticas descriptivas de columnas numéricas
println("=== Estadísticas descriptivas ===")
dfCursos.describe("horas", "valoracion").show()

=== Estadísticas descriptivas ===
+-------+-----------------+------------------+
|summary|            horas|        valoracion|
+-------+-----------------+------------------+
|  count|                5|                 5|
|   mean|             88.0|               4.7|
| stddev|41.47288270665544|0.1581138830084192|
|    min|               40|               4.5|
|    max|              140|               4.9|
+-------+-----------------+------------------+



🔹 Ejercicio 2 — Cargar un CSV con inferencia de schema

id,nombre,edad,departamento,salario,activo
1,Ana García,28,Ingeniería,42000,true
2,Luis Martínez,34,Marketing,38000,true
3,Marta López,22,Ingeniería,35000,true
4,Pedro Ruiz,41,Dirección,75000,true
5,Carmen Díaz,29,Marketing,36500,true
6,Jorge Santos,38,Ingeniería,48000,false
7,Elena Vega,31,RRHH,33000,true
8,Tomás Gil,45,Dirección,82000,true
9,Laura Prieto,26,Ingeniería,39000,true
10,Andrés Mora,33,Marketing,41000,false

In [46]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val ruta = "C:/Notebooks/datos"
Files.createDirectories(Paths.get(ruta))

val empleadoCSV = 
"""id,nombre,edad,departamento,salario,activo
1,Ana García,28,Ingeniería,42000,true
2,Luis Martínez,34,Marketing,38000,true
3,Marta López,22,Ingeniería,35000,true
4,Pedro Ruiz,41,Dirección,75000,true
5,Carmen Díaz,29,Marketing,36500,true
6,Jorge Santos,38,Ingeniería,48000,false
7,Elena Vega,31,RRHH,33000,true
8,Tomás Gil,45,Dirección,82000,true
9,Laura Prieto,26,Ingeniería,39000,true
10,Andrés Mora,33,Marketing,41000,false
"""

val pathCSV = s"$ruta/empleados.csv"

Files.write(
  Paths.get(pathCSV),
  empleadoCSV.getBytes(StandardCharsets.UTF_8)
)

println(s"CSV creado en: $pathCSV")

CSV creado en: C:/Notebooks/datos/empleados.csv


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
ruta: String = "C:/Notebooks/datos"
res46_3: java.nio.file.Path = C:\Notebooks\datos
empleadoCSV: String = """id,nombre,edad,departamento,salario,activo
1,Ana García,28,Ingeniería,42000,true
2,Luis Martínez,34,Marketing,38000,true
3,Marta López,22,Ingeniería,35000,true
4,Pedro Ruiz,41,Dirección,75000,true
5,Carmen Díaz,29,Marketing,36500,true
6,Jorge Santos,38,Ingeniería,48000,false
7,Elena Vega,31,RRHH,33000,true
8,Tomás Gil,45,Dirección,82000,true
9,Laura Prieto,26,Ingeniería,39000,true
10,Andrés Mora,33,Marketing,41000,false
"""
pathCSV: String = "C:/Notebooks/datos/empleados.csv"
res46_6: java.nio.file.Path = C:\Notebooks\datos\empleados.csv

In [49]:
val dfEmpleados = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("C:/Notebooks/datos/empleados.csv")

println("=== Primeras filas ===")
dfEmpleados.show()

println("=== Schema inferido ===")
dfEmpleados.printSchema()

=== Primeras filas ===
+---+-------------+----+------------+-------+------+
| id|       nombre|edad|departamento|salario|activo|
+---+-------------+----+------------+-------+------+
|  1|   Ana García|  28|  Ingeniería|  42000|  true|
|  2|Luis Martínez|  34|   Marketing|  38000|  true|
|  3|  Marta López|  22|  Ingeniería|  35000|  true|
|  4|   Pedro Ruiz|  41|   Dirección|  75000|  true|
|  5|  Carmen Díaz|  29|   Marketing|  36500|  true|
|  6| Jorge Santos|  38|  Ingeniería|  48000| false|
|  7|   Elena Vega|  31|        RRHH|  33000|  true|
|  8|    Tomás Gil|  45|   Dirección|  82000|  true|
|  9| Laura Prieto|  26|  Ingeniería|  39000|  true|
| 10|  Andrés Mora|  33|   Marketing|  41000| false|
+---+-------------+----+------------+-------+------+

=== Schema inferido ===
root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- departamento: string (nullable = true)
 |-- salario: integer (nullable = true)
 |-- activo

dfEmpleados: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 4 more fields]

In [50]:
// Ahora definimos el schema manualmente
val schemaEmpleados = StructType(List(
  StructField("id",           IntegerType, nullable = false),
  StructField("nombre",       StringType,  nullable = true),
  StructField("edad",         IntegerType, nullable = true),
  StructField("departamento", StringType,  nullable = true),
  StructField("salario",      DoubleType,  nullable = true),  // Double en vez de Int
  StructField("activo",       BooleanType, nullable = true)
))

val dfEmpleadosTyped = spark.read
  .option("header", "true")
  .schema(schemaEmpleados)
  .csv("C:/Notebooks/datos/empleados.csv")

println("=== Schema manual aplicado ===")
dfEmpleadosTyped.printSchema()

// Diferencia: el campo salario ahora es DoubleType
// Útil si luego vamos a calcular medias o porcentajes

=== Schema manual aplicado ===
root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- departamento: string (nullable = true)
 |-- salario: double (nullable = true)
 |-- activo: boolean (nullable = true)



schemaEmpleados: StructType = Seq(
  StructField(
    name = "id",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "nombre",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "edad",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "departamento",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "salario",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "activo",
    dataType = BooleanType,
    nullable = true,
    metadata = {}
  )
)
dfEmpleadosTyped: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 4 more fields]

In [51]:
println("=== Tipos de cada columna ===")
dfEmpleadosTyped.dtypes.foreach { case (col, tipo) =>
  println(f"  $col%-15s → $tipo")
}

println(s"\nTotal empleados: ${dfEmpleadosTyped.count()}")

println("\n=== Estadísticas de edad y salario ===")
dfEmpleadosTyped.describe("edad", "salario").show()

=== Tipos de cada columna ===
  id              → IntegerType
  nombre          → StringType
  edad            → IntegerType
  departamento    → StringType
  salario         → DoubleType
  activo          → BooleanType

Total empleados: 10

=== Estadísticas de edad y salario ===
+-------+------------------+-----------------+
|summary|              edad|          salario|
+-------+------------------+-----------------+
|  count|                10|               10|
|   mean|              32.7|          46950.0|
| stddev|7.0561242115547325|17211.83378441188|
|    min|                22|          33000.0|
|    max|                45|          82000.0|
+-------+------------------+-----------------+



Practica 3

In [56]:
val dfProductos = spark.read
  .option("multiline", "true")
  .json("C:/Notebooks/datos/empleados.json")

println("=== Productos cargados ===")
dfProductos.show(truncate = false)

println("=== Schema del JSON ===")
dfProductos.printSchema()

=== Productos cargados ===
+--------------+---+-------------------+-------+-----+
|categoria     |id |nombre             |precio |stock|
+--------------+---+-------------------+-------+-----+
|Informatica   |101|Laptop Pro         |1299.99|45   |
|Perifericos   |102|Teclado Inalámbrico|59.9   |120  |
|Monitores     |103|Monitor 27"        |349.0  |30   |
|Perifericos   |104|Ratón Óptico       |24.95  |200  |
|Audio         |105|Auriculares USB    |89.5   |75   |
|Perifericos   |106|Webcam HD          |79.0   |60   |
|Almacenamiento|107|Disco SSD 1TB      |119.99 |90   |
|Perifericos   |108|Hub USB-C          |44.9   |150  |
+--------------+---+-------------------+-------+-----+

=== Schema del JSON ===
root
 |-- categoria: string (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- precio: double (nullable = true)
 |-- stock: long (nullable = true)



dfProductos: org.apache.spark.sql.package.DataFrame = [categoria: string, id: bigint ... 3 more fields]

In [57]:
println(s"Total de productos: ${dfProductos.count()}")

println("\n=== Categorías disponibles ===")
dfProductos.select("categoria").distinct().show()

println("=== Rango de precios ===")
dfProductos.describe("precio", "stock").show()

Total de productos: 8

=== Categorías disponibles ===
+--------------+
|     categoria|
+--------------+
|   Perifericos|
|   Informatica|
|     Monitores|
|Almacenamiento|
|         Audio|
+--------------+

=== Rango de precios ===
+-------+----------------+-----------------+
|summary|          precio|            stock|
+-------+----------------+-----------------+
|  count|               8|                8|
|   mean|       258.40375|            96.25|
| stddev|433.007817248389|57.36786058910885|
|    min|           24.95|               30|
|    max|         1299.99|              200|
+-------+----------------+-----------------+



In [58]:
println(s"Total de productos: ${dfProductos.count()}")

println("\n=== Categorías disponibles ===")
dfProductos.select("categoria").distinct().show()

println("=== Rango de precios ===")
dfProductos.describe("precio", "stock").show()

Total de productos: 8

=== Categorías disponibles ===
+--------------+
|     categoria|
+--------------+
|   Perifericos|
|   Informatica|
|     Monitores|
|Almacenamiento|
|         Audio|
+--------------+

=== Rango de precios ===
+-------+----------------+-----------------+
|summary|          precio|            stock|
+-------+----------------+-----------------+
|  count|               8|                8|
|   mean|       258.40375|            96.25|
| stddev|433.007817248389|57.36786058910885|
|    min|           24.95|               30|
|    max|         1299.99|              200|
+-------+----------------+-----------------+



🔹 Ejercicio 4 — Comparar inferencia vs. schema manual en JSON

In [59]:
// Schema manual: controlamos que id y stock sean Int, no Long
val schemaProductos = StructType(List(
  StructField("id",        IntegerType, nullable = false),
  StructField("nombre",    StringType,  nullable = true),
  StructField("categoria", StringType,  nullable = true),
  StructField("precio",    DoubleType,  nullable = true),
  StructField("stock",     IntegerType, nullable = true)
))

val dfProductosTyped = spark.read
  .option("multiline", "true")
  .schema(schemaProductos)
  .json("C:/Notebooks/datos/productos.json")

println("=== Schema controlado ===")
dfProductosTyped.printSchema()

println("\n=== Comparación de tipos ===")
println("Con inferSchema:")
dfProductos.dtypes.foreach { case (c, t) => println(s"  $c → $t") }

println("\nCon schema manual:")
dfProductosTyped.dtypes.foreach { case (c, t) => println(s"  $c → $t") }

=== Schema controlado ===
root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- precio: double (nullable = true)
 |-- stock: integer (nullable = true)


=== Comparación de tipos ===
Con inferSchema:
  categoria → StringType
  id → LongType
  nombre → StringType
  precio → DoubleType
  stock → LongType

Con schema manual:
  id → IntegerType
  nombre → StringType
  categoria → StringType
  precio → DoubleType
  stock → IntegerType


schemaProductos: StructType = Seq(
  StructField(
    name = "id",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "nombre",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "categoria",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "precio",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "stock",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  )
)
dfProductosTyped: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 3 more fields]

In [60]:
// show() tiene tres variantes útiles
println("show() por defecto — 20 filas, textos truncados a 20 chars:")
dfEmpleados.show()

show() por defecto — 20 filas, textos truncados a 20 chars:
+---+-------------+----+------------+-------+------+
| id|       nombre|edad|departamento|salario|activo|
+---+-------------+----+------------+-------+------+
|  1|   Ana García|  28|  Ingeniería|  42000|  true|
|  2|Luis Martínez|  34|   Marketing|  38000|  true|
|  3|  Marta López|  22|  Ingeniería|  35000|  true|
|  4|   Pedro Ruiz|  41|   Dirección|  75000|  true|
|  5|  Carmen Díaz|  29|   Marketing|  36500|  true|
|  6| Jorge Santos|  38|  Ingeniería|  48000| false|
|  7|   Elena Vega|  31|        RRHH|  33000|  true|
|  8|    Tomás Gil|  45|   Dirección|  82000|  true|
|  9| Laura Prieto|  26|  Ingeniería|  39000|  true|
| 10|  Andrés Mora|  33|   Marketing|  41000| false|
+---+-------------+----+------------+-------+------+



In [61]:
println("show(3) — solo las primeras 3 filas:")
dfEmpleados.show(3)

show(3) — solo las primeras 3 filas:
+---+-------------+----+------------+-------+------+
| id|       nombre|edad|departamento|salario|activo|
+---+-------------+----+------------+-------+------+
|  1|   Ana García|  28|  Ingeniería|  42000|  true|
|  2|Luis Martínez|  34|   Marketing|  38000|  true|
|  3|  Marta López|  22|  Ingeniería|  35000|  true|
+---+-------------+----+------------+-------+------+
only showing top 3 rows


In [62]:
println("show(3, truncate = false) — 3 filas, textos completos:")
dfEmpleados.show(3, truncate = false)

show(3, truncate = false) — 3 filas, textos completos:
+---+-------------+----+------------+-------+------+
|id |nombre       |edad|departamento|salario|activo|
+---+-------------+----+------------+-------+------+
|1  |Ana García   |28  |Ingeniería  |42000  |true  |
|2  |Luis Martínez|34  |Marketing   |38000  |true  |
|3  |Marta López  |22  |Ingeniería  |35000  |true  |
+---+-------------+----+------------+-------+------+
only showing top 3 rows


In [63]:
// columns devuelve un Array[String]
println(s"\nColumnas del DataFrame de empleados:")
println(dfEmpleados.columns.mkString(" | "))
// id | nombre | edad | departamento | salario | activo


Columnas del DataFrame de empleados:
id | nombre | edad | departamento | salario | activo


🏢 Caso de Estudio Propuesto : AgroData Cooperativa

In [64]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val ruta = "C:/Notebooks/datos"
Files.createDirectories(Paths.get(ruta))

val parcelaCSV = 
"""id_parcela,provincia,municipio,superficie_ha,cultivo,año_alta,en_produccion,rendimiento_kg_ha
P001,Sevilla,Carmona,12.5,Naranja,2015,true,28000
P002,Huelva,Lepe,8.3,Fresa,2018,true,45000
P003,Almería,Níjar,25.0,Tomate,2012,true,85000
P004,Sevilla,Écija,6.7,Aceituna,2009,false,3200
P005,Murcia,Totana,15.2,Limón,2016,true,22000
P006,Almería,El Ejido,30.1,Pimiento,2014,true,62000
P007,Huelva,Moguer,9.8,Fresa,2020,true,41000
P008,Murcia,Lorca,18.4,Melocotón,2011,false,9500
P009,Sevilla,Utrera,22.0,Naranja,2013,true,31000
P010,Almería,Vícar,11.6,Pepino,2019,true,74000
P011,Murcia,Alhama,7.9,Limón,2017,true,20500
P012,Huelva,Cartaya,14.3,Fresa,2015,true,43000
P013,Sevilla,Marchena,5.2,Aceituna,2010,false,2900
P014,Almería,Roquetas,28.7,Tomate,2011,true,88000
P015,Murcia,Mazarrón,16.5,Pimiento,2018,true,58000
"""

val pathCSV = s"$ruta/parcelas.csv"

Files.write(
  Paths.get(pathCSV),
  parcelaCSV.getBytes(StandardCharsets.UTF_8)
)

println(s"CSV creado en: $pathCSV")

CSV creado en: C:/Notebooks/datos/parcelas.csv


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
ruta: String = "C:/Notebooks/datos"
res64_3: java.nio.file.Path = C:\Notebooks\datos
parcelaCSV: String = """id_parcela,provincia,municipio,superficie_ha,cultivo,año_alta,en_produccion,rendimiento_kg_ha
P001,Sevilla,Carmona,12.5,Naranja,2015,true,28000
P002,Huelva,Lepe,8.3,Fresa,2018,true,45000
P003,Almería,Níjar,25.0,Tomate,2012,true,85000
P004,Sevilla,Écija,6.7,Aceituna,2009,false,3200
P005,Murcia,Totana,15.2,Limón,2016,true,22000
P006,Almería,El Ejido,30.1,Pimiento,2014,true,62000
P007,Huelva,Moguer,9.8,Fresa,2020,true,41000
P008,Murcia,Lorca,18.4,Melocotón,2011,false,9500
P009,Sevilla,Utrera,22.0,Naranja,2013,true,31000
P010,Almería,Vícar,11.6,Pepino,2019,true,74000
P011,Murcia,Alhama,7.9,Limón,2017,true,20500
P012,Huelva,Cartaya,14.3,Fresa,2015,true,43000
P013,Sevilla,Marchena,5.2,Aceituna,2010,false,2900
P014,Almería,Roquetas,28.7,Tomate,2011,true,88000
P015,Murcia,Mazarrón,16.5,Pimiento,2018,true,58000


In [65]:
val jsonProducto = 
"""
[
  {
    "codigo": "NAR",
    "nombre": "Naranja",
    "familia": "Citrico",
    "precio_mercado_euro_kg": 0.45,
    "demanda_exportacion": "Alta",
    "certificacion_eco": false
  },
  {
    "codigo": "FRE",
    "nombre": "Fresa",
    "familia": "Baya",
    "precio_mercado_euro_kg": 2.10,
    "demanda_exportacion": "Muy Alta",
    "certificacion_eco": true
  },
  {
    "codigo": "TOM",
    "nombre": "Tomate",
    "familia": "Hortaliza",
    "precio_mercado_euro_kg": 0.85,
    "demanda_exportacion": "Alta",
    "certificacion_eco": false
  },
  {
    "codigo": "ACE",
    "nombre": "Aceituna",
    "familia": "Oleaginosa",
    "precio_mercado_euro_kg": 0.60,
    "demanda_exportacion": "Media",
    "certificacion_eco": true
  },
  {
    "codigo": "LIM",
    "nombre": "Limón",
    "familia": "Citrico",
    "precio_mercado_euro_kg": 0.55,
    "demanda_exportacion": "Alta",
    "certificacion_eco": false
  },
  {
    "codigo": "PIM",
    "nombre": "Pimiento",
    "familia": "Hortaliza",
    "precio_mercado_euro_kg": 1.20,
    "demanda_exportacion": "Muy Alta",
    "certificacion_eco": true
  },
  {
    "codigo": "PEP",
    "nombre": "Pepino",
    "familia": "Hortaliza",
    "precio_mercado_euro_kg": 0.70,
    "demanda_exportacion": "Media",
    "certificacion_eco": false
  },
  {
    "codigo": "MEL",
    "nombre": "Melocotón",
    "familia": "Drupa",
    "precio_mercado_euro_kg": 1.35,
    "demanda_exportacion": "Media",
    "certificacion_eco": false
  }
]
"""

val ruta = "C:/Notebooks/datos" // Usa tu propia ruta, sino te dará error
Files.createDirectories(Paths.get(ruta))

val rutaProducto = s"$ruta/productos.json"

Files.write(
  Paths.get(rutaProducto),
  jsonProducto.getBytes(StandardCharsets.UTF_8)
)

println(s"JSON multilínea creado en: $rutaProducto")

JSON multilínea creado en: C:/Notebooks/datos/productos.json


jsonProducto: String = """
[
  {
    "codigo": "NAR",
    "nombre": "Naranja",
    "familia": "Citrico",
    "precio_mercado_euro_kg": 0.45,
    "demanda_exportacion": "Alta",
    "certificacion_eco": false
  },
  {
    "codigo": "FRE",
    "nombre": "Fresa",
    "familia": "Baya",
    "precio_mercado_euro_kg": 2.10,
    "demanda_exportacion": "Muy Alta",
    "certificacion_eco": true
  },
  {
    "codigo": "TOM",
    "nombre": "Tomate",
    "familia": "Hortaliza",
    "precio_mercado_euro_kg": 0.85,
    "demanda_exportacion": "Alta",
    "certificacion_eco": false
  },
  {
    "codigo": "ACE",
    "nombre": "Aceituna",
    "familia": "Oleaginosa",
    "precio_mercado_euro_kg": 0.60,
    "demanda_exportacion": "Media",
    "certificacion_eco": true
  },
  {
    "codigo": "LIM",
    "nombre": "Limón",
    "familia": "Citrico",
    "precio_mercado_euro_kg": 0.55,
...
ruta: String = "C:/Notebooks/datos"
res65_2: java.nio.file.Path = C:\Notebooks\datos
rutaProducto: String = "C:/Notebooks/

Tarea 1 — Inicialización del entorno

In [66]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.types._

val spark = SparkSession.builder()
  .appName("AgroData_Analisis")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

import spark.implicits._   // necesario para .toDF() y otras conversiones
import org.apache.spark.sql.types._

spark.sparkContext.setLogLevel("ERROR")

println(s"✅ Spark ${spark.version} listo")

✅ Spark 4.1.1 listo


import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.types._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@a9c4c1c
import spark.implicits._
import org.apache.spark.sql.types._

Tarea 2 — Carga del CSV de parcelas con inferSchema

Lo que debes hacer:

* Cargar parcelas.csv con header = true e inferSchema = true
* Mostrar las primeras 5 filas con show(5)
* Imprimir el schema completo con printSchema()
* Imprimir el número total de parcelas registradas con count()
* Listar los nombres de todas las columnas con columns

In [67]:
val dfParcelas = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("C:/Notebooks/datos/parcelas.csv")

println("=== Primeras filas ===")
dfParcelas.show(5)

println("=== Schema inferido ===")
dfParcelas.printSchema()

println(s"Total parcelas: ${dfParcelas.count()}")

println(s"\nColumnas del DataFrame de parcelas:")
println(dfParcelas.columns.mkString(" | "))

=== Primeras filas ===
+----------+---------+---------+-------------+--------+--------+-------------+-----------------+
|id_parcela|provincia|municipio|superficie_ha| cultivo|año_alta|en_produccion|rendimiento_kg_ha|
+----------+---------+---------+-------------+--------+--------+-------------+-----------------+
|      P001|  Sevilla|  Carmona|         12.5| Naranja|    2015|         true|            28000|
|      P002|   Huelva|     Lepe|          8.3|   Fresa|    2018|         true|            45000|
|      P003|  Almería|    Níjar|         25.0|  Tomate|    2012|         true|            85000|
|      P004|  Sevilla|    Écija|          6.7|Aceituna|    2009|        false|             3200|
|      P005|   Murcia|   Totana|         15.2|   Limón|    2016|         true|            22000|
+----------+---------+---------+-------------+--------+--------+-------------+-----------------+
only showing top 5 rows
=== Schema inferido ===
root
 |-- id_parcela: string (nullable = true)
 |-- prov

dfParcelas: org.apache.spark.sql.package.DataFrame = [id_parcela: string, provincia: string ... 6 more fields]

Tarea 3 — Definir el schema manualmente y recargar

El responsable ha revisado el schema inferido y señala dos problemas:

* superficie_ha debería ser DoubleType — ✅ correcto tal cual
* rendimiento_kg_ha debería ser DoubleType (no IntegerType) para poder hacer cálculos de medias con decimales
* año_alta debería ser IntegerType — ✅ correcto tal cual
* id_parcela es un código alfanumérico que nunca debería ser nulo: nullable = false

Lo que debes hacer:

* Definir un StructType con los ocho campos corrigiendo los puntos anteriores
* Recargar el CSV usando .schema(...) en lugar de inferSchema
* Imprimir el nuevo schema con printSchema()
* Imprimir los tipos con dtypes en formato columna → tipo para que la dirección pueda comparar fácilmente

In [73]:
// Ahora definimos el schema manualmente
val schemaParcelas = StructType(List(
  StructField("id_parcela",      IntegerType, nullable = false),
  StructField("provincia",       StringType,  nullable = true),
  StructField("municipio",       StringType,  nullable = true),
  StructField("superficie_ha",   DoubleType,  nullable = true),
  StructField("cultivo",         StringType,  nullable = true),
  StructField("anho_alta",        IntegerType, nullable = true),
  StructField("en_produccion",   BooleanType, nullable = true),
  StructField("rendimiento_kg_ha", DoubleType, nullable = true)
))

// Recarga el CSV
val dfParcelasTyped = spark.read
  .option("header", "true")
  .schema(schemaParcelas)
  .csv("C:/Notebooks/datos/parcelas.csv")

// Imprime el nuevo schema
println("=== Schema manual aplicado ===")
dfParcelasTyped.printSchema()

// Imprimir los tipos con dtypes
println("=== Tipos de cada columna ===")
dfParcelasTyped.dtypes.foreach { case (col, tipo) =>
  println(f"  $col%-15s → $tipo")
}


=== Schema manual aplicado ===
root
 |-- id_parcela: integer (nullable = true)
 |-- provincia: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- superficie_ha: double (nullable = true)
 |-- cultivo: string (nullable = true)
 |-- anho_alta: integer (nullable = true)
 |-- en_produccion: boolean (nullable = true)
 |-- rendimiento_kg_ha: double (nullable = true)

=== Tipos de cada columna ===
  id_parcela      → IntegerType
  provincia       → StringType
  municipio       → StringType
  superficie_ha   → DoubleType
  cultivo         → StringType
  anho_alta       → IntegerType
  en_produccion   → BooleanType
  rendimiento_kg_ha → DoubleType


schemaParcelas: StructType = Seq(
  StructField(
    name = "id_parcela",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "provincia",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "municipio",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "superficie_ha",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "cultivo",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "anho_alta",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "en_produccion",
...
dfParcelasTyped: org.apache.spark.sql.package.DataFrame = [id_parcela: int, provincia: string ... 6 more fields]

Tarea 4 — Primer informe estadístico de las parcelas

La directora de operaciones necesita un resumen numérico rápido de las parcelas para la reunión de mañana.

Lo que debes hacer:

* Usar describe() sobre las columnas superficie_ha y rendimiento_kg_ha
* Usar describe() también sobre año_alta
* Imprimir los resultados con un título descriptivo

In [74]:
println("====Descripcion de superficie y rencimiento")
dfParcelasTyped.describe("superficie_ha","rendimiento_kg_ha").show()


====Descripcion de superficie y rencimiento
+-------+------------------+------------------+
|summary|     superficie_ha| rendimiento_kg_ha|
+-------+------------------+------------------+
|  count|                15|                15|
|   mean|15.479999999999999|40873.333333333336|
| stddev| 7.931240220077275|27911.479120008433|
|    min|               5.2|            2900.0|
|    max|              30.1|           88000.0|
+-------+------------------+------------------+



In [77]:
println("====Descripcion de fecha de alta")
dfParcelasTyped.describe("anho_alta").show()

====Descripcion de fecha de alta
+-------+------------------+
|summary|         anho_alta|
+-------+------------------+
|  count|                15|
|   mean|2014.5333333333333|
| stddev|3.4613512362879963|
|    min|              2009|
|    max|              2020|
+-------+------------------+

